# CNN-only baseline theo từng dataset

Notebook này chạy baseline CNN-only với cùng tiền xử lý và cùng chiến lược chia dữ liệu như `cnn_lstm/CNN_LSTM.py`.

Thiết kế thí nghiệm:
- Train riêng trên `kaggle`, `csic`, `obfuscation`.
- Mỗi model có tokenizer riêng, fit trên train split của chính dataset đó.
- Sau khi train, mỗi model được đánh giá trên toàn bộ test split của các dataset.
- Kết quả dùng để so sánh công bằng với CNN-LSTM theo hướng mới của đề tài.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "cnn_only":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT / "cnn_only")

Path.cwd()

## 1. Kiểm tra nhanh dữ liệu sau tiền xử lý

In [ ]:
from preprocessing import preprocess_data as prep

datasets = prep.load_clean_datasets(prep.KAGGLE_PATH, prep.CSIC_PATH, prep.OBFU_PATH)
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("rows:", len(df))
    print("label counts:", df["label"].value_counts().sort_index().to_dict())
    print("length p95:", float(df["payload"].str.len().quantile(0.95)))

## 2. Smoke test nhanh

Chạy cell này trước để kiểm tra pipeline. Cell này chỉ lấy sample nhỏ, không dùng làm kết quả báo cáo.

In [ ]:
%run train_cnn_only.py --sample-size 3000 --obfu-sample-size 3000 --epochs 2 --train-sources kaggle csic obfuscation --output-dir artifacts_by_dataset_smoke

## 3. Chạy thật cho một dataset

Khi muốn tiết kiệm thời gian, nên chạy từng nguồn một. Ví dụ dưới đây train CNN-only trên Kaggle, rồi test trên Kaggle/CSIC/Obfuscation.

In [ ]:
%run train_cnn_only.py --train-sources kaggle --epochs 50 --output-dir artifacts_by_dataset

## 4. Chạy đầy đủ cả 3 nguồn

Cell này sẽ train 3 model CNN-only riêng biệt. Thời gian chạy sẽ lâu hơn nhiều.

In [ ]:
# %run train_cnn_only.py --train-sources all --epochs 50 --output-dir artifacts_by_dataset

## 5. Đọc bảng kết quả cross-evaluation

In [ ]:
import pandas as pd

result_path = Path("artifacts_by_dataset/cross_eval_results.csv")
if result_path.exists():
    results = pd.read_csv(result_path)
    display(results)
else:
    print("Chưa có kết quả. Hãy chạy cell train ở trên trước.")

In [ ]:
matrix_path = Path("artifacts_by_dataset/cross_eval_accuracy_matrix.csv")
if matrix_path.exists():
    display(pd.read_csv(matrix_path, index_col=0))
else:
    print("Chưa có accuracy matrix.")